# March Mania · Women’s temporal-change replication

**Milestone 07 · Preserve → replicate unchanged → inspect → test components → checkpoint**

The women’s change-only representation lowered Brier in 2017 and 2019. Those discovery years selected the family and are **excluded from the next gate**. The additional years 2016 and 2018 are also previously-used project history, not untouched tests.

This notebook defines **no new features** and makes **no new rating fits**. It reuses seven base and seven temporal snapshots, replays six existing classifiers, and fits only two replication classifiers. At most eight component fits follow if the prespecified gate passes. The failed shooting and schedule-record families remain unpromoted.

Use **Python (March Mania)**. Run the terminal tests from START_HERE.md first. No pip installation, AWS API calls, Git writes, or submissions are involved.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink

KIT = Path.cwd().resolve()
if not (KIT / 'run_round07.py').is_file():
    KIT = Path.home() / 'march_temporal_validation'
assert (KIT / 'run_round07.py').is_file(), 'Open this notebook inside march_temporal_validation.'
sys.path.insert(0, str(KIT))
from run_round07 import run_stage
from validation_plots import figures
pio.renderers.default = 'plotly_mimetype'
print('Kernel:', sys.executable)
print('Research folder:', KIT)
print('No feature/rating rebuild. Two replication fits; components are gated.')

## 1. Preserve the discovery evidence

Negative `delta_vs_anchor` means lower Brier. The 2019 change-only gain is small and its **log loss worsened**; Brier improvement is not improvement on every diagnostic. Do not confuse these historical women-only results with the combined 2026 submission score.

In [ ]:
prior = pd.read_csv(KIT / 'evidence' / 'metrics.csv', float_precision='round_trip')
view = prior.query("Gender == 'W'")[['Season','recipe','brier','log_loss','delta_vs_anchor']].copy()
for col in ['brier','log_loss','delta_vs_anchor']:
    view[col] = view[col].map(lambda x: f'{x:.7f}')
display(view)
print('These are the returned round-06 results; no new fit has run here.')

## 2. Verify and replay before spending compute

The existing repository, raw hashes, known edited notebooks, source definitions and environment must match the returned evidence. All six reused classifiers are checked against exact saved predictions and returned Brier values. The two edited canonical notebooks remain untouched. A missing cache stops; it never triggers a hidden rebuild.

Preparation limit: **180 seconds**; heartbeat: **15 seconds**.

In [ ]:
run_stage('prepare', max_seconds=180)
record = json.loads((KIT / 'reports/latest_run.json').read_text())
RUN = Path(record['run_dir'])
print(json.dumps(json.loads((RUN / 'prepare.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'prior_replay.csv')[['Season','recipe','brier','source']])
display(pd.read_csv(RUN / 'feature_registry.csv').query("family == 'late_change'")[['feature','description']])

## 3. Two new replication classifiers

| Validation | Training | Anchor | Anchor + two changes |
|---|---|---|---|
| 2016 | 2013–2015 | Replay | **New fit** |
| 2017 | 2013–2016 | Replay | Replay discovery |
| 2018 | 2013–2017 | Replay | **New fit** |
| 2019 | 2013–2018 | Replay | Replay discovery |

Fixed logistic C=0.1, mirrored orientations, training-only RMS scaling, physical-game weighting and input definitions are unchanged. No window tuning, feature selection sweep, ensemble or calibration changes. Evaluation limit: **120 seconds**.

In [ ]:
run_stage('replicate', max_seconds=120)
metrics = pd.read_csv(RUN / 'replication_metrics.csv', float_precision='round_trip')
view = metrics[['Season','recipe','role','brier','delta_vs_anchor','source']].copy()
for col in ['brier','delta_vs_anchor']:
    view[col] = view[col].map(lambda x: f'{x:.7f}')
display(view)
gate = json.loads((RUN / 'gate.json').read_text())
print(json.dumps(gate, indent=2))

## 4. Gated component ablations

Both additional years (2016 and 2018) must improve and their mean Brier delta must be **≤ −0.0005**. This is a compute-allocation rule, not significance or automatic selection. Discovery years 2017/2019 do not enter it.

If the gate passes, drop offensive change and drop defensive change separately across all four years (eight fits maximum). The other inputs remain fixed. Positive `delta_vs_full` means removing the named feature worsened Brier. Both original and additional years appear in component diagnostics, so they are post-selection exploration.

`SKIPPED_BY_GATE` is a valid outcome. Continue to the report cells. Component stage limit: **180 seconds**.

In [ ]:
run_stage('components', max_seconds=180)
print(json.dumps(json.loads((RUN / 'component_receipt.json').read_text()), indent=2))
components = pd.read_csv(RUN / 'component_metrics.csv')
if len(components):
    display(components[['Season','removed_feature','brier','delta_vs_full','delta_vs_anchor']])
else:
    print('Components skipped by the gate. The unchanged family is not promoted.')

## 5. Interactive evidence

Ten plots summarize discovery, per-season comparisons, paired loss changes, confidence ranges, calibration, input support, training-only overlap, coefficients and season sensitivity. A passed gate adds two component plots. No confidence interval or causal interpretation is claimed from a few reused seasons.

In [ ]:
plots = figures(RUN, KIT / 'evidence')
assert len(plots) in (10, 12)
for fig in plots[:5]:
    fig.show()

In [ ]:
for fig in plots[5:]:
    fig.show()

## 6. Export, preserve, and stop

Reporting verifies prior and raw bytes again. The small return archive excludes raw rows, per-game predictions, fitted models and private notebook edits. Models and intermediate evidence stay in the new fingerprinted `private_runs` folder. Reporting limit: **120 seconds**.

Save this notebook, download `reports/milestone_07_return.zip`, and attach it in ChatGPT. No further experiment or Git operation launches automatically.

In [ ]:
run_stage('report', max_seconds=120)
summary = json.loads((RUN / 'summary.json').read_text())
print(json.dumps(summary, indent=2))
report = json.loads((KIT / 'reports/latest_report.json').read_text())
print('Return ZIP:', report['return_zip'])
print('Interactive HTML:', report['html'])
display(FileLink(str(Path(report['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(report['html']).relative_to(KIT))))
print('Save with Ctrl+S. Stop this milestone here.')